# Claude 3.7 Sonnet의 병렬 tool 호출


Claude 3.7 Sonnet은 `disable_parallel_tool_use`를 설정하지 않았어도 응답에서 병렬 tool 호출을 할 가능성이 더 낮을 수 있습니다. 이를 해결하기 위해 다른 tool 호출을 동시에 감싸는 meta-tool 역할의 "batch tool"을 도입하기를 권장합니다. 이 tool이 있으면 model이 이를 사용하여 여러 tool을 병렬로 동시에 호출하는 것을 확인했습니다.

문제를 살펴보고 이 우회 방법을 더 자세히 검토하겠습니다.


In [ ]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## 여러 tool 호출로 query 수행하기

Claude에 병렬 tool 호출을 허용하는 것이 기본 동작임을 기억하세요. 기본 `tool_choice`인 `auto`와 함께, 이는 지정한 tool 중 어떤 것이든 호출하거나 단일 assistant turn에서 둘 이상을 호출할 수 있음을 뜻합니다.

Claude에 `get_weather`와 `get_time` tool을 설정하겠습니다.


In [3]:
def get_weather(location):
    # 날씨를 가져오는 것처럼 하고 고정값만 반환합니다.
    return f"The weather in {location} is 72 degrees and sunny."


def get_time(location):
    # 시간을 가져오는 것처럼 하고 고정값만 반환합니다.
    return f"The time in {location} is 12:32 PM."


weather_tool = {
    "name": "get_weather",
    "description": "Gets the weather for in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
        },
        "required": ["location"],
    },
}

time_tool = {
    "name": "get_time",
    "description": "Gets the time in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
        },
        "required": ["location"],
    },
}


def process_tool_call(tool_name, tool_input):
    if tool_name == "get_weather":
        return get_weather(tool_input["location"])
    elif tool_name == "get_time":
        return get_time(tool_input["location"])
    else:
        raise ValueError(f"Unexpected tool name: {tool_name}")

다음으로 Claude에 이 tool을 제공하고 query를 수행하겠습니다.


In [4]:
def make_query_and_print_result(messages, tools=None):
    response = client.messages.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=1000,
        tool_choice={"type": "auto"},
        tools=tools or [weather_tool, time_tool],
    )

    for block in response.content:
        match block.type:
            case "text":
                print(block.text)
            case "tool_use":
                print(f"Tool: {block.name}({block.input})")
            case _:
                raise ValueError(f"Unexpected block type: {block.type}")

    return response


MESSAGES = [{"role": "user", "content": "What's the weather and time in San Francisco?"}]

response = make_query_and_print_result(MESSAGES)

I'll check the current weather and time in San Francisco for you.
Tool: get_weather({'location': 'San Francisco, CA'})


둘 다 요청했는데도 Claude가 날씨의 단일 tool 호출만 반환한 것을 보셨나요?

날씨 tool을 호출하여 계속하면 어떻게 되는지 보겠습니다.


In [5]:
last_tool_call = response.content[1]

MESSAGES.append({"role": "assistant", "content": response.content})
MESSAGES.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": last_tool_call.id,
                "content": process_tool_call(response.content[1].name, response.content[1].input),
            }
        ],
    }
)

response = make_query_and_print_result(MESSAGES)

Tool: get_time({'location': 'San Francisco, CA'})


이제 Claude가 시간을 가져오는 두 번째 tool 호출을 했습니다. 기술적으로는 즉시 일어났지만 "왕복"이 필요했으므로 낭비일 수 있습니다. 먼저 날씨를 요청하여 처리해야 했고, _그다음_ 시간을 요청하여 이제 _그것을_ 처리해야 합니다.

Claude는 여전히 결과를 올바르게 사용하지만, 둘 다 한 번에 사용하도록 유도하면 동시에 처리할 수 있어 이로울 수 있습니다.


## batch tool 도입하기

여러 tool 호출을 하나로 결합하는 데 사용할 기회를 주도록 `batch_tool`을 도입하겠습니다.


In [6]:
import json

batch_tool = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "types": "object",
                    "properties": {
                        "name": {
                            "types": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "types": "string",
                            "description": "The arguments to the tool",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


def process_tool_with_maybe_batch(tool_name, tool_input):
    if tool_name == "batch_tool":
        results = []
        for invocation in tool_input["invocations"]:
            results.append(
                process_tool_call(invocation["name"], json.loads(invocation["arguments"]))
            )
        return "\n".join(results)
    else:
        return process_tool_call(tool_name, tool_input)

이제 기존 날씨 및 시간 tool과 새 batch tool을 함께 제공하여 날씨와 시간을 요구하는 query에서 어떻게 되는지 보겠습니다.


In [7]:
MESSAGES = [{"role": "user", "content": "What's the weather and time in San Francisco?"}]

response = make_query_and_print_result(MESSAGES, tools=[weather_tool, time_tool, batch_tool])

I can help you check both the weather and the time in San Francisco. Let me get that information for you right away.
Tool: batch_tool({'invocations': [{'name': 'get_weather', 'arguments': '{"location": "San Francisco, CA"}'}, {'name': 'get_time', 'arguments': '{"location": "San Francisco, CA"}'}]})


이번에는 Claude가 batch tool로 시간과 날씨를 한 번에 query한 점을 보세요. 동시에 처리할 수 있으므로 결과까지의 전체 latency를 개선할 수 있습니다.


In [8]:
last_tool_call = response.content[1]

MESSAGES.append({"role": "assistant", "content": response.content})
MESSAGES.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": last_tool_call.id,
                "content": process_tool_with_maybe_batch(
                    response.content[1].name, response.content[1].input
                ),
            }
        ],
    }
)

response = make_query_and_print_result(MESSAGES)

Here's the information you requested:

Weather in San Francisco, CA: 72 degrees and sunny
Time in San Francisco, CA: 12:32 PM

Is there anything else you'd like to know about San Francisco?
